# Ablation study B: no human-intent input, and flat MAPPO without the hierarchy (3 seeds each)

Settings: **Accelerator: None**, **Internet: On**. Run cells 1-2 to check, then **Save Version -> Save & Run All (Commit)** once (about 8 hours). Download **ablation_B.zip** from the Output tab.

In [ ]:
!pip install -q mujoco pettingzoo gymnasium
!git clone -q https://github.com/swarnasmita/marl-edu-platform.git
%cd marl-edu-platform
import os
print(os.cpu_count(), 'CPU cores')

In [ ]:
# quick check (about 1 minute): should end with a line starting with 'done;'
!python agents/train_mappo.py --ablation no_intent --seed 99 --out /kaggle/working/test_run --iterations 2 --episodes-per-iter 4 --eval-every 2 --val-scenarios 2 --workers 4
!rm -rf /kaggle/working/test_run

In [ ]:
# train: final MAPPO with 'no_intent' switched off, 3 seeds (about 75 minutes each)
for seed in (0, 1, 2):
    !python agents/train_mappo.py --ablation no_intent --seed {seed} --out /kaggle/working/runs/mappo_no_intent_seed{seed} --workers 4

In [ ]:
# test on the 100 unseen scenarios x 3 simulated humans
!python agents/evaluate.py --workers 4 --label "MAPPO without human-intent input (hierarchical)" --out /kaggle/working/results/ablations/no_intent --checkpoints /kaggle/working/runs/mappo_no_intent_seed0/best.pt /kaggle/working/runs/mappo_no_intent_seed1/best.pt /kaggle/working/runs/mappo_no_intent_seed2/best.pt

In [ ]:
# flat MAPPO (no hierarchy): moves chosen every step from raw observations, 3 seeds (about 60-80 minutes each)
for seed in (0, 1, 2):
    !python agents/train_flat.py --seed {seed} --out /kaggle/working/runs/flat_seed{seed} --workers 4

In [ ]:
# test flat MAPPO on the same 100 unseen scenarios x 3 simulated humans
!python agents/train_flat.py --workers 4 --out /kaggle/working/results/ablations/flat --evaluate /kaggle/working/runs/flat_seed0/best.pt /kaggle/working/runs/flat_seed1/best.pt /kaggle/working/runs/flat_seed2/best.pt

In [ ]:
# one zip to download
!cd /kaggle/working && zip -qr ablation_B.zip results runs/mappo_no_intent_seed0/best.pt runs/mappo_no_intent_seed0/log.csv runs/mappo_no_intent_seed1/best.pt runs/mappo_no_intent_seed1/log.csv runs/mappo_no_intent_seed2/best.pt runs/mappo_no_intent_seed2/log.csv runs/flat_seed0/best.pt runs/flat_seed0/log.csv runs/flat_seed1/best.pt runs/flat_seed1/log.csv runs/flat_seed2/best.pt runs/flat_seed2/log.csv